In [1]:
# 0 Importaciones
# 0.1 Importar librerias
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) # Para ver todas las columnas

In [2]:
# 0.2. Importamos los dataframes que necesitamos de la carpeta interim (que tenian un concat)
b_add = pd.read_csv('../Data/interim/b_add_interim.csv')
df_customer = pd.read_csv('../Data/interim/df_customer_interim.csv')

In [3]:
# Limpieza de datos
#  1 Cambiar tipo de datos

##  Cambiamos aquellas columnas con valores 0.0 y 1.0 a un formato unificado 'yes' y 'no' igual que en la columna 'y'
b_add[['default', 'housing', 'loan']].value_counts(dropna=False) # Podemos apreciar como hay valores nan
b_add[['default', 'housing', 'loan', 'y']] = b_add[['default', 'housing', 'loan', 'y']].replace({0.0: 'no', 1.0: 'yes'})

# Cambiamos tipo de datos

## A tipo de datos categoricos
b_add[['default', 'housing', 'loan', 'poutcome', 'contact', 'job', 'marital', 'education']] = b_add[['default', 'housing', 'loan', 'poutcome', 'contact','job', 'marital', 'education']].astype('category')

## A float
cols = ['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in cols:
    b_add[col] = b_add[col].str.replace(',', '.')

b_add[cols] = b_add[cols].astype(float)

## columna date, en tipo de dato fecha, pero antes hay que arreglar como pandas detecta las fechas
# diccionario con fechas de español a ingles
meses = {
    'enero': 'January',
    'febrero': 'February',
    'marzo': 'March',
    'abril': 'April',
    'mayo': 'May',
    'junio': 'June',
    'julio': 'July',
    'agosto': 'August',
    'septiembre': 'September',
    'octubre': 'October',
    'noviembre': 'November',
    'diciembre': 'December'
}

# Cambiamos a float la columna income
df_customer['Income'] = df_customer['Income'].astype(float)



# Cambiamos cada fila de fecha usando el diccionario creado previamente usando un metodo de string
for esp, eng in meses.items():
    b_add['date'] = b_add['date'].str.replace(esp, eng, regex=False)

# Convertimos el string previamente formateado en fecha
b_add['date'] = pd.to_datetime(b_add['date'], errors='coerce')


In [4]:
# 2. Rellenar nulos en job, marital y education con 'unknown'
cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']

for col in cols:
    # Añadimos la categoría 'unknown' a cada columna (Series categórica)
    # cat.add_categories es un método de pandas (Series), no de DataFrame
    b_add[col] = b_add[col].cat.add_categories('unknown')
    
    # Rellenamos los valores nulos con la nueva categoría
    b_add[col] = b_add[col].fillna('unknown')

In [5]:
# 2.1 Rellenar 'age' con mediana
b_add['age'] = b_add['age'].fillna(b_add['age'].median())

# Cambiamos el tipo de datos de float a int en la columba 'age' 
b_add['age'] = b_add['age'].astype(int)

In [6]:
# 2.2 Usamos ffill (forward fill) para imputar los valores nulos en variables macroeconómicas como
# 'cons.price.idx' y 'euribor3m'. Este método rellena los valores faltantes con el último valor
# válido disponible, asumiendo que estas variables cambian de forma gradual y continua en el tiempo.
# Es una estrategia adecuada en este caso porque son indicadores económicos de baja variabilidad
# y evita introducir valores artificiales (como la media o interpolaciones más agresivas) que podrían
# distorsionar la estructura temporal de los datos.


b_add['cons.price.idx'] = b_add['cons.price.idx'].ffill()
b_add['euribor3m'] = b_add['euribor3m'].ffill()

In [7]:
# 2.3 
b_add[['age', 'cons.price.idx', 'euribor3m', 'date']].describe(include=['int', 'float', 'datetime']).T

# Podemos ver que tenemos 248 valores nan en fecha. Veremos que haremos con ello despues en el merge

,count,mean,min,25%,50%,75%,max,std
age,43000.0,39.741698,17.0,33.0,38.0,46.0,98.0,9.817735
cons.price.idx,43000.0,93.573913,92.201,93.075,93.749,93.994,94.767,0.579669
euribor3m,43000.0,3.613673,0.634,1.344,4.857,4.961,5.045,1.736894
date,42752,2017-07-01 19:55:11.676646,2015-01-01 00:00:00,2016-04-01 00:00:00,2017-07-04 00:00:00,2018-10-01 06:00:00,2019-12-31 00:00:00,NaN


In [8]:
# 3. Merge
# Para antes de unir los 2 datasets. Vamos a renombrar la columna "id_" para que ambos datasets tengan el mismo nombre de columna
b_add.rename(columns={"id_": "ID"}, inplace=True)
b_add.head()

# La columna que queremos analizar es "y", entonces queremos que salgan todos los registros de b_add
df = b_add.merge(right=df_customer, how ='left', on= "ID")

# Informacion basica del nuevo df
print(df.shape) # (43000 filas, 28 columnas)
print(df.columns) # Columnas del nuevo df
print(f"Nulos: {df.isnull().sum()[df.isnull().sum() > 0]}") # Nulos
print(f"Numero de duplicados: {df.duplicated().sum()}") # Duplicados

(43000, 28)
Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m',
       'nr.employed', 'y', 'date', 'latitude', 'longitude', 'ID', 'Income',
       'Kidhome', 'Teenhome', 'Dt_Customer', 'NumWebVisitsMonth'],
      dtype='str')
Nulos: date    248
dtype: int64
Numero de duplicados: 0


In [9]:
# 4. Eliminacion de columnas que no son relevantes para explicar "y" o que no dan valor
df = df.drop(columns = ['ID', 'latitude', 'longitude','previous', 'Dt_Customer', 'pdays'])

# Se elimina la variable 'pdays' porque contiene un valor sentinela (999) que indica ausencia de información sobre contactos previos.
# Dado que su interpretación no es directa en el contexto del análisis actual y puede generar confusión,
# se decide prescindir de ella.


In [10]:
# 5. Mandar a la carpeta prepared
df.to_csv('../Data/cleaned/df_cleaned.csv', index = False) # index = False para que no se cree columna "Unnamed: 0"
